# Feature Engineering & Data Preparation for Machine Learning

## Kronos Intelligence Portfolio Project

### Student Performance Classification

---

### Project Goal

The objective of this notebook is to prepare the cleaned student performance dataset for machine learning by selecting predictive features, preventing data leakage, encoding categorical variables, scaling numerical features where appropriate, and creating reproducible training and testing datasets.

The resulting datasets will serve as the foundation for developing classification models in the next phase of the project.

---

# Project Objectives

This notebook focuses on preparing the dataset for predictive modeling.

The objectives are to:

- Select appropriate predictive features.
- Identify and remove variables that could introduce data leakage.
- Encode categorical variables into numerical representations.
- Separate predictor variables from the target variable.
- Create training and testing datasets.
- Apply feature scaling where appropriate.
- Save modeling-ready datasets for reuse.
- Document every preprocessing decision to ensure reproducibility and transparency.

In [1]:
# ==========================================================
# Import Required Libraries
# ==========================================================

import os

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

from google.colab import drive

In [2]:
# ==========================================================
# Project Environment Setup
# ==========================================================

drive.mount('/content/drive')

PROJECT_ROOT = "/content/drive/MyDrive/Kronos Intelligence/Student Performance Classification"

RAW_DATA = os.path.join(PROJECT_ROOT, "data", "raw")
PROCESSED_DATA = os.path.join(PROJECT_ROOT, "data", "processed")
NOTEBOOKS = os.path.join(PROJECT_ROOT, "notebooks")

print("✅ Project environment ready.")

Mounted at /content/drive
✅ Project environment ready.


In [3]:
# ==========================================================
# Load Clean Dataset
# ==========================================================

clean_df = pd.read_csv(
    os.path.join(PROCESSED_DATA, "students_clean.csv")
)

clean_df.head()

,student_id,age,gender,ethnicity,parental_education,family_income,school_type,school_region,study_hours_per_week,attendance_rate,...,has_laptop,sleep_hours,stress_level,motivation_score,reading_score,writing_score,math_score,science_score,overall_gpa,passed
0,STU00000,15,M,B,phd,low,private,urban,2.0,91.7,...,yes,8.2,3.3,1.8,70.7,58.3,88.9,74.5,2.80,1
1,STU00001,23,F,A,phd,low,private,urban,14.2,76.8,...,yes,5.4,4.9,5.8,66.1,83.9,94.5,83.6,3.29,1
2,STU00002,22,F,B,bachelor,middle,public,suburban,3.3,83.1,...,yes,5.5,3.5,6.1,51.3,71.8,78.9,71.2,2.93,1
3,STU00003,19,M,D,some_college,high,public,suburban,17.9,72.8,...,yes,5.3,8.5,7.2,100.0,95.7,82.9,69.4,3.34,1
4,STU00004,19,M,C,some_college,middle,public,suburban,12.9,83.0,...,yes,7.5,1.0,4.5,87.5,73.0,88.5,74.8,3.27,1


# Review of Notebook 03

The exploratory data analysis identified several variables that demonstrated meaningful relationships with academic performance.

Key findings included:

- Study hours showed the strongest behavioral relationship with overall GPA.
- Attendance and motivation displayed moderate positive relationships with academic success.
- Higher stress levels were associated with lower GPAs.
- Sleep hours demonstrated a modest positive relationship with academic performance.
- Academic assessment scores exhibited strong correlations with GPA but require evaluation for potential data leakage before model development.

These findings provide the foundation for selecting features appropriate for predictive modeling.

# Feature Engineering Strategy

The goal of feature engineering is to transform the cleaned dataset into a format suitable for machine learning while preserving predictive information and minimizing bias.

This notebook will:

- Identify the prediction target.
- Evaluate predictor variables.
- Remove features that introduce data leakage.
- Encode categorical variables.
- Scale numerical features where appropriate.
- Create reproducible training and testing datasets.

Every transformation will be documented to ensure transparency and reproducibility.

# Identify the Target Variable

## Business Question

**What outcome is the model designed to predict?**

The objective of this project is to develop a classification model that predicts whether a student will successfully pass based on information that could reasonably be collected during the academic term.

The target variable for this project is **`passed`**, where:

- **1 = Student Passed**
- **0 = Student Did Not Pass**

Selecting the target variable is the first step in preparing the dataset for supervised machine learning.

In [4]:
# ==========================================================
# Define Target Variable
# ==========================================================

target = "passed"

print(f"Target Variable: {target}")

Target Variable: passed


# Feature Selection

## Business Question

**Which variables should be used to predict student success?**

Not every variable in the dataset should be included in the predictive model. Feature selection involves identifying variables that provide useful predictive information while excluding those that introduce redundancy or information that would not be available at prediction time.

The goal is to develop a model that can identify at-risk students early enough for meaningful intervention.

In [5]:
# ==========================================================
# Available Features
# ==========================================================

clean_df.columns.tolist()

['student_id',
 'age',
 'gender',
 'ethnicity',
 'parental_education',
 'family_income',
 'school_type',
 'school_region',
 'study_hours_per_week',
 'attendance_rate',
 'extracurricular_activities',
 'sports_participation',
 'tutoring_sessions',
 'parental_involvement',
 'internet_access',
 'has_laptop',
 'sleep_hours',
 'stress_level',
 'motivation_score',
 'reading_score',
 'writing_score',
 'math_score',
 'science_score',
 'overall_gpa',
 'passed']

# Data Leakage Assessment

## Business Question

**Which variables would introduce data leakage if included in the predictive model?**

Data leakage occurs when a model is trained using information that would not be available at the time predictions are made. While these variables may improve model accuracy during training, they produce unrealistic performance and reduce the model's usefulness in real-world applications.

For an early intervention model, only information that could reasonably be collected before final academic outcomes should be included.

In [6]:
# ==========================================================
# Variables Evaluated for Data Leakage
# ==========================================================

potential_leakage = [
    "overall_gpa",
    "reading_score",
    "writing_score",
    "math_score",
    "science_score"
]

print("Potential leakage variables:")

for column in potential_leakage:
    print(f"- {column}")

Potential leakage variables:
- overall_gpa
- reading_score
- writing_score
- math_score
- science_score


## Leakage Decision

The following variables will **not** be used as predictors:

- Overall GPA
- Reading Score
- Writing Score
- Math Score
- Science Score

These variables represent academic outcomes that are typically measured after coursework has been completed. Because the objective is to identify at-risk students early in the academic term, including these variables would introduce data leakage and produce unrealistically optimistic model performance.

Removing these variables ensures that the resulting model more closely reflects a real-world early warning system capable of supporting proactive intervention strategies.

In [7]:
# ==========================================================
# Remove Leakage Variables
# ==========================================================

model_df = clean_df.drop(
    columns=[
        "overall_gpa",
        "reading_score",
        "writing_score",
        "math_score",
        "science_score"
    ]
)

model_df.head()

,student_id,age,gender,ethnicity,parental_education,family_income,school_type,school_region,study_hours_per_week,attendance_rate,extracurricular_activities,sports_participation,tutoring_sessions,parental_involvement,internet_access,has_laptop,sleep_hours,stress_level,motivation_score,passed
0,STU00000,15,M,B,phd,low,private,urban,2.0,91.7,5,no,17,medium,yes,yes,8.2,3.3,1.8,1
1,STU00001,23,F,A,phd,low,private,urban,14.2,76.8,0,yes,15,high,yes,yes,5.4,4.9,5.8,1
2,STU00002,22,F,B,bachelor,middle,public,suburban,3.3,83.1,4,yes,16,high,yes,yes,5.5,3.5,6.1,1
3,STU00003,19,M,D,some_college,high,public,suburban,17.9,72.8,1,yes,6,medium,yes,yes,5.3,8.5,7.2,1
4,STU00004,19,M,C,some_college,middle,public,suburban,12.9,83.0,1,yes,2,medium,no,yes,7.5,1.0,4.5,1


In [8]:
# ==========================================================
# Remaining Features
# ==========================================================

print(f"Original Features: {clean_df.shape[1]}")
print(f"Model Features: {model_df.shape[1]}")

model_df.columns.tolist()

Original Features: 25
Model Features: 20


['student_id',
 'age',
 'gender',
 'ethnicity',
 'parental_education',
 'family_income',
 'school_type',
 'school_region',
 'study_hours_per_week',
 'attendance_rate',
 'extracurricular_activities',
 'sports_participation',
 'tutoring_sessions',
 'parental_involvement',
 'internet_access',
 'has_laptop',
 'sleep_hours',
 'stress_level',
 'motivation_score',
 'passed']

# Encode Categorical Variables

## Business Question

**How can categorical variables be transformed into a format suitable for machine learning?**

Most machine learning algorithms require numerical input. Because this dataset contains several categorical variables (such as gender, ethnicity, school type, and parental education), these variables must be converted into numerical features before model training.

For this project, **One-Hot Encoding** is used because it preserves all categories without introducing artificial ordinal relationships between values.

This approach improves model interpretability while ensuring compatibility with a wide range of classification algorithms.

In [9]:
# ==========================================================
# Identify Categorical Variables
# ==========================================================

categorical_columns = model_df.select_dtypes(include="object").columns.tolist()

print("Categorical Variables:\n")

for column in categorical_columns:
    print(f"- {column}")

Categorical Variables:

- student_id
- gender
- ethnicity
- parental_education
- family_income
- school_type
- school_region
- sports_participation
- parental_involvement
- internet_access
- has_laptop


In [10]:
# ==========================================================
# One-Hot Encode Categorical Variables
# ==========================================================

encoded_df = pd.get_dummies(
    model_df,
    columns=categorical_columns,
    drop_first=True
)

encoded_df.head()

,age,study_hours_per_week,attendance_rate,extracurricular_activities,tutoring_sessions,sleep_hours,stress_level,motivation_score,passed,student_id_STU00001,...,family_income_middle,school_type_private,school_type_public,school_region_suburban,school_region_urban,sports_participation_yes,parental_involvement_low,parental_involvement_medium,internet_access_yes,has_laptop_yes
0,15,2.0,91.7,5,17,8.2,3.3,1.8,1,False,...,False,True,False,False,True,False,False,True,True,True
1,23,14.2,76.8,0,15,5.4,4.9,5.8,1,True,...,False,True,False,False,True,True,False,False,True,True
2,22,3.3,83.1,4,16,5.5,3.5,6.1,1,False,...,True,False,True,True,False,True,False,False,True,True
3,19,17.9,72.8,1,6,5.3,8.5,7.2,1,False,...,False,False,True,True,False,True,False,True,True,True
4,19,12.9,83.0,1,2,7.5,1.0,4.5,1,False,...,True,False,True,True,False,True,False,True,False,True


In [11]:
# ==========================================================
# Compare Dataset Dimensions
# ==========================================================

print(f"Original Dataset Shape : {model_df.shape}")

print(f"Encoded Dataset Shape  : {encoded_df.shape}")

Original Dataset Shape : (10000, 20)
Encoded Dataset Shape  : (10000, 10030)


## Encoding Summary

The categorical variables were successfully transformed using **One-Hot Encoding**.

This process increased the number of predictor variables because each category was converted into its own binary indicator variable.

Using One-Hot Encoding prevents machine learning algorithms from incorrectly assuming that categorical values have a natural numerical order while preserving the information contained within each category.

The encoded dataset is now suitable for use with common classification algorithms such as Logistic Regression, Random Forest, Gradient Boosting, and XGBoost.

# Engineered Feature Inventory

The final modeling dataset contains the following groups of predictor variables:

### Demographic Features

- Age
- Gender
- Ethnicity

### Family Characteristics

- Family Income
- Parental Education
- Parental Involvement

### School Characteristics

- School Type
- School Region

### Behavioral Features

- Study Hours
- Attendance Rate
- Motivation Score
- Stress Level
- Sleep Hours
- Tutoring Sessions
- Extracurricular Activities
- Sports Participation

### Technology Access

- Internet Access
- Laptop Availability

Categorical variables were transformed using One-Hot Encoding to produce the final machine learning feature set.

# Separate Features and Target

## Business Question

**Which variables will be used as predictors, and which variable will the model learn to predict?**

Machine learning models require the predictor variables (**X**) to be separated from the target variable (**y**).

This separation establishes the inputs that the model will use to learn patterns associated with student success.

In [12]:
# ==========================================================
# Separate Features and Target
# ==========================================================

X = encoded_df.drop(columns=["passed"])

y = encoded_df["passed"]

print("Feature Matrix Shape:", X.shape)
print("Target Shape:", y.shape)

Feature Matrix Shape: (10000, 10029)
Target Shape: (10000,)


In [13]:
# ==========================================================
# Review Feature Matrix
# ==========================================================

X.head()

,age,study_hours_per_week,attendance_rate,extracurricular_activities,tutoring_sessions,sleep_hours,stress_level,motivation_score,student_id_STU00001,student_id_STU00002,...,family_income_middle,school_type_private,school_type_public,school_region_suburban,school_region_urban,sports_participation_yes,parental_involvement_low,parental_involvement_medium,internet_access_yes,has_laptop_yes
0,15,2.0,91.7,5,17,8.2,3.3,1.8,False,False,...,False,True,False,False,True,False,False,True,True,True
1,23,14.2,76.8,0,15,5.4,4.9,5.8,True,False,...,False,True,False,False,True,True,False,False,True,True
2,22,3.3,83.1,4,16,5.5,3.5,6.1,False,True,...,True,False,True,True,False,True,False,False,True,True
3,19,17.9,72.8,1,6,5.3,8.5,7.2,False,False,...,False,False,True,True,False,True,False,True,True,True
4,19,12.9,83.0,1,2,7.5,1.0,4.5,False,False,...,True,False,True,True,False,True,False,True,False,True


In [14]:
# ==========================================================
# Review Target Variable
# ==========================================================

y.head()

,passed
0,1
1,1
2,1
3,1
4,1


# Feature Scaling Strategy

## Business Question

**Should the numerical features be scaled before model development?**

Feature scaling standardizes numerical variables so that they contribute proportionally during model training.

Scaling is particularly important for distance-based and gradient-based algorithms such as Logistic Regression, Support Vector Machines, and Neural Networks.

Tree-based algorithms such as Decision Trees, Random Forests, and Gradient Boosting generally do not require feature scaling.

To support multiple machine learning algorithms in later notebooks, numerical features will be standardized using **StandardScaler** after the training and testing datasets have been created.

# Train-Test Split

## Business Question

**How should the dataset be divided for machine learning model development and evaluation?**

To objectively evaluate model performance, the dataset is divided into separate training and testing sets.

- **Training Set:** Used to train the machine learning model.
- **Testing Set:** Reserved for evaluating how well the model generalizes to new, unseen data.

A **stratified train-test split** is used to preserve the proportion of students who passed and failed in both datasets. This is particularly important because the target variable is imbalanced.

In [15]:
# ==========================================================
# Create Train-Test Split
# ==========================================================

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"Training Features : {X_train.shape}")
print(f"Testing Features  : {X_test.shape}")
print(f"Training Target   : {y_train.shape}")
print(f"Testing Target    : {y_test.shape}")

Training Features : (8000, 10029)
Testing Features  : (2000, 10029)
Training Target   : (8000,)
Testing Target    : (2000,)


## Verify Class Distribution

Using stratified sampling helps ensure that the training and testing datasets maintain approximately the same proportion of students who passed and failed.

Preserving the class distribution improves the reliability of model evaluation and reduces the risk of biased performance estimates.

In [16]:
# ==========================================================
# Verify Target Distribution
# ==========================================================

print("Training Set")

print(
    y_train.value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nTesting Set")

print(
    y_test.value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Training Set
passed
1    96.1
0     3.9
Name: proportion, dtype: float64

Testing Set
passed
1    96.1
0     3.9
Name: proportion, dtype: float64


# Feature Scaling

## Business Question

**How can numerical variables be standardized for machine learning?**

Many machine learning algorithms perform better when numerical variables are measured on a similar scale.

To prevent data leakage, the scaler is fitted **only on the training data** and then applied to both the training and testing datasets.

This ensures that information from the testing dataset does not influence the training process.

In [17]:
# ==========================================================
# Scale Features
# ==========================================================

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

print("Scaling Complete")

Scaling Complete


In [18]:
# ==========================================================
# Convert Scaled Arrays to DataFrames
# ==========================================================

X_train_scaled = pd.DataFrame(
    X_train_scaled,
    columns=X_train.columns,
    index=X_train.index
)

X_test_scaled = pd.DataFrame(
    X_test_scaled,
    columns=X_test.columns,
    index=X_test.index
)

X_train_scaled.head()

,age,study_hours_per_week,attendance_rate,extracurricular_activities,tutoring_sessions,sleep_hours,stress_level,motivation_score,student_id_STU00001,student_id_STU00002,...,family_income_middle,school_type_private,school_type_public,school_region_suburban,school_region_urban,sports_participation_yes,parental_involvement_low,parental_involvement_medium,internet_access_yes,has_laptop_yes
3280,-1.573231,-0.398130,0.314335,-0.888851,-0.655383,0.179905,-0.906001,-0.710764,0.0,-0.011181,...,-1.001501,-0.504293,0.736624,-0.819477,1.233721,-0.81969,1.723434,-0.890234,0.435827,0.538043
1519,0.012488,-0.009548,0.994640,-0.888851,0.167911,1.019065,0.029897,-0.114209,0.0,-0.011181,...,0.998501,1.982973,-1.357545,-0.819477,-0.810556,-0.81969,-0.580237,-0.890234,0.435827,0.538043
2614,-1.256087,0.395929,1.710751,-0.304081,-0.326065,1.270813,-1.529933,0.780623,0.0,-0.011181,...,0.998501,-0.504293,0.736624,-0.819477,1.233721,-0.81969,-0.580237,1.123300,0.435827,0.538043
821,0.646775,-0.279866,0.529168,-1.473621,1.649839,0.347737,0.289868,-0.064496,0.0,-0.011181,...,-1.001501,-0.504293,-1.357545,1.220291,-0.810556,-0.81969,-0.580237,1.123300,0.435827,0.538043
8758,-0.938943,-0.685343,-0.234683,-0.888851,1.155863,0.347737,-0.282069,1.675455,0.0,-0.011181,...,0.998501,1.982973,-1.357545,-0.819477,1.233721,-0.81969,-0.580237,-0.890234,0.435827,0.538043


# Save Modeling Datasets

The prepared datasets are saved for reuse in subsequent notebooks.

Saving intermediate datasets promotes reproducibility, improves workflow efficiency, and ensures that every modeling notebook uses the same training and testing data.

In [20]:
# ==========================================================
# Save the Fitted StandardScaler
# ==========================================================

import joblib

joblib.dump(
    scaler,
    os.path.join(MODEL_DATA, "standard_scaler.pkl")
)

print("✅ StandardScaler saved successfully.")

✅ StandardScaler saved successfully.


In [19]:
# ==========================================================
# Save Modeling Datasets
# ==========================================================

MODEL_DATA = os.path.join(PROJECT_ROOT, "data", "model")

os.makedirs(MODEL_DATA, exist_ok=True)

X_train_scaled.to_csv(
    os.path.join(MODEL_DATA, "X_train.csv"),
    index=False
)

X_test_scaled.to_csv(
    os.path.join(MODEL_DATA, "X_test.csv"),
    index=False
)

y_train.to_csv(
    os.path.join(MODEL_DATA, "y_train.csv"),
    index=False
)

y_test.to_csv(
    os.path.join(MODEL_DATA, "y_test.csv"),
    index=False
)

print("✅ Modeling datasets saved successfully.")

✅ Modeling datasets saved successfully.


In [21]:
# ==========================================================
# Save Feature Names
# ==========================================================

feature_names = pd.DataFrame({
    "Feature": X.columns
})

feature_names.to_csv(
    os.path.join(MODEL_DATA, "feature_names.csv"),
    index=False
)

print("✅ Feature names saved successfully.")

✅ Feature names saved successfully.


# Summary of Engineered Features

The feature engineering process successfully prepared the dataset for machine learning.

The following preprocessing steps were completed:

- Selected the target variable.
- Removed variables that could introduce data leakage.
- Applied One-Hot Encoding to categorical variables.
- Separated predictor variables from the target.
- Created reproducible training and testing datasets using stratified sampling.
- Standardized numerical features using StandardScaler.
- Saved modeling-ready datasets for reuse.

The resulting datasets provide a reliable foundation for developing and evaluating classification models.

# Executive Insights

The feature engineering process transformed the cleaned dataset into a machine learning-ready format while preserving reproducibility and minimizing the risk of data leakage.

Several important preprocessing decisions were made:

- Variables representing final academic outcomes were removed to prevent data leakage.
- Categorical variables were successfully encoded using One-Hot Encoding.
- Predictor variables and the target variable were separated for supervised learning.
- The dataset was divided into stratified training and testing sets to preserve the original class distribution.
- Numerical features were standardized using StandardScaler to support algorithms that are sensitive to feature magnitude.

The resulting datasets provide a reliable foundation for developing and comparing multiple classification models in the next phase of the project.

# Data Governance Considerations

Feature engineering plays an important role in ensuring that predictive models are both technically sound and ethically responsible.

Key governance considerations for this project include:

- Removing variables that introduce data leakage to prevent unrealistic model performance.
- Documenting all preprocessing decisions to support transparency and reproducibility.
- Applying consistent preprocessing to both training and testing datasets.
- Protecting sensitive educational information through appropriate data handling practices.
- Ensuring that future predictive models are regularly evaluated for fairness, bias, and performance across different student populations.

These practices align with responsible data governance principles and support the development of trustworthy predictive analytics solutions.

# Key Preprocessing Decisions

The following preprocessing decisions were made during feature engineering:

- Selected **passed** as the prediction target.
- Removed overall GPA and individual assessment scores to prevent data leakage.
- Applied One-Hot Encoding to all categorical variables.
- Preserved numerical variables for modeling.
- Used an 80/20 stratified train-test split.
- Standardized numerical features using StandardScaler after splitting the data.
- Saved modeling datasets for reuse in subsequent notebooks.

These decisions support the development of robust and reproducible machine learning models.

# Notebook Summary

This notebook prepared the student performance dataset for predictive modeling by performing feature engineering and data preprocessing.

Major accomplishments included:

- Selecting appropriate predictor variables.
- Preventing data leakage.
- Encoding categorical features.
- Separating predictors and target variables.
- Creating reproducible training and testing datasets.
- Standardizing numerical features.
- Saving modeling-ready datasets for future analysis.

The processed datasets are now ready for machine learning model development and evaluation.

# Next Steps

The next notebook will focus on **Machine Learning Model Development**.

The following classification algorithms will be trained and compared:

- Logistic Regression
- Decision Tree
- Random Forest
- Gradient Boosting
- XGBoost (if available)

Each model will be evaluated using consistent performance metrics, including:

- Accuracy
- Precision
- Recall
- F1-Score
- ROC-AUC
- Confusion Matrix

The objective is to identify the model that provides the best balance between predictive performance and interpretability while supporting early identification of students who may benefit from academic intervention.

---

# Kronos Intelligence Portfolio Progress

| Notebook | Status |
|----------|--------|
| 01 — Data Understanding | ✅ Complete |
| 02 — Data Cleaning | ✅ Complete |
| 03 — Exploratory Data Analysis | ✅ Complete |
| 04 — Feature Engineering & Data Preparation | ✅ Complete |
| 05 — Machine Learning Model Development | ⏳ Next |
| 06 — Model Evaluation & Comparison | 🔜 Planned |
| 07 — Business Recommendations & Final Report | 🔜 Planned |

---

**Prepared by**

# Kronos Intelligence

*Data Analytics • Predictive Analytics • Data Governance*